In [1]:
import sys; sys.path.append("..")   # lets the notebook import from src/
import pandas as pd
from src.data.load import load_data
from src.evaluation.nested_cv import run_nested_cv
from src.evaluation.metrics import summarize, per_fold, subject_level

X, y, groups = load_data("../data/raw/parkinsons.data")
print(X.shape, y.value_counts().to_dict(), groups.nunique())   # expect (195, 20), {1:147, 0:48}, 32

results, params = {}, {}
for name in ["logreg", "svm_rbf", "knn", "rf"]:
    oof, best = run_nested_cv(name, X, y, groups)
    results[name], params[name] = oof, best

pd.DataFrame({n: summarize(o) for n, o in results.items()}).T.round(3)

(195, 20) {1: 147, 0: 48} 32


,accuracy,balanced_acc,sensitivity_PD,specificity_healthy,precision_PD,f1_PD,roc_auc,pr_auc_healthy
logreg,0.749,0.742,0.755,0.729,0.895,0.819,0.842,0.623
svm_rbf,0.692,0.712,0.673,0.750,0.892,0.767,0.807,0.501
knn,0.800,0.685,0.912,0.458,0.838,0.873,0.746,0.485
rf,0.723,0.648,0.796,0.500,0.830,0.812,0.769,0.497


In [2]:
for n, o in results.items():
    subj, cm, ties = subject_level(o)
    print(n); print(per_fold(o).round(3)); print(params[n]); print(cm, "ties:", ties)

logreg
                      mean    std
accuracy             0.752  0.138
balanced_acc         0.719  0.094
sensitivity_PD       0.755  0.220
specificity_healthy  0.683  0.253
precision_PD         0.899  0.083
f1_PD                0.802  0.149
roc_auc              0.830  0.134
pr_auc_healthy       0.649  0.269
   fold  inner_auc  model__C select__k
0     1   0.786161      0.01       all
1     2   0.825629     10.00         8
2     3   0.812977      0.01        12
3     4   0.857183      0.01        12
4     5   0.838949      0.01       all
[[ 5  3]
 [ 4 20]] ties: 4
svm_rbf
                      mean    std
accuracy             0.696  0.169
balanced_acc         0.717  0.153
sensitivity_PD       0.667  0.199
specificity_healthy  0.767  0.224
precision_PD         0.885  0.139
f1_PD                0.752  0.151
roc_auc              0.813  0.109
pr_auc_healthy       0.623  0.215
   fold  inner_auc  model__C  model__gamma select__k
0     1   0.795724       0.1          0.01       all
1     